# Import Dependencies

In [ ]:
# !pip install mesa --quiet

In [67]:
try:
    import mesa
except:
    %pip install mesa --quiet
    import mesa
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Resource Classes

In [122]:
class Sugar(mesa.Agent):
    ''' 
    Sugar
        - contains an amount of sugar
        - grows one amount of sugar for each turn
    '''

    def __init__(self, unique_id, model, pos, max_sugar):
        super().__init__(unique_id, model)
        self.initial_pos = pos # pos refers to position. Mesa’s newer versions track self.pos internally, and adding the line self.pos = pos manually creates inconsistency. That's why we have replaced here to self.initial_pos 
        self.amount = max_sugar
        self.max_sugar = max_sugar

    def step(self):
        '''
        Sugar growth function, adds one unit of sugar each step until max amount
        '''
        self.amount = min([self.max_sugar, self.amount+1])

class Spice(mesa.Agent):
    '''
    Spice:
        - contains an amount of spice
        - grows one amount of spice at each turn
    '''

    def __init__(self, unique_id, model, pos, max_spice):
        super().__init__(unique_id, model)
        self.initial_pos = pos
        self.amount = max_spice # This tells us how much spice at a given time step
        self.max_spice = max_spice # This tells the most that can be at a location

    def step(self):
        '''
        Spice growth function, adds one unit of spice each step until max amount
        '''
        self.amount = min([self.max_spice, self.amount+1])

# Trader Class

In [ ]:
class Trader(mesa.Agent):
    '''
    Trader:
        - has a metabolism for sugar and spice
        - harvest and traders sugar and spice to survive and thrive
    '''

    def __init__(self, unique_id, model, pos, moore=False, sugar=0, spice=0, metabolism_sugar=0, metabolism_spice=0, vision=0):
        super().__init__(unique_id, model)

        # self.pos=pos # Removed to avoid duplicating setting the position and avoid Warning message
        self.moore=moore
        self.sugar=sugar
        self.spice=spice
        self.metabolism_sugar=metabolism_sugar
        self.metabolism_spice=metabolism_spice
        self.vision=vision


# Model Class

In [ ]:
class SugarscapeG1mt(mesa.Model):
    '''
    A model class to manage Sugarscape with Traders (G1mt)
    from Growing Artificial Societies by Axtell and Epstein
    '''

    def __init__(self, width=50, height=50, initial_population=200, endowment_min=25, endowment_max=50, metabolism_min=1, metabolism_max=5, vision_min=1, vision_max=5):
        
        super().__init__()
        # Iniciate width and height of sugarscape
        self.width=width
        self.height=height
        # Iniciate population attributes
        self.initial_population=initial_population
        self.endowment_min=endowment_min
        self.endowment_max=endowment_max
        self.metabolism_min=metabolism_min
        self.metabolism_max=metabolism_max
        self.vision_min=vision_min
        self.vision_max=vision_max

        # Iniciate mesa grid class
        self.grid=mesa.space.MultiGrid(self.width, self.height, torus=False)

        # Read in Landscape file from supplementary material
        sugar_distribution=np.genfromtxt("sugar-map.txt")
        spice_distribution=np.flip(sugar_distribution, 1)
        # plt.imshow(spice_distribution, origin="lower")

        # Iniciate Scheduler -> track what order our agents are activated
        self.schedule = mesa.time.RandomActivationByType(self)

        # Instanciate each agent
        agent_id = 0
        for _,(x,y) in self.grid.coord_iter():
            max_sugar = sugar_distribution[x,y] # define the sugar from the sugar distribution
            if max_sugar > 0:
                sugar = Sugar(agent_id, self, (x,y), max_sugar) # we pass self and the 'self' will be translated into an agent 'model' when it goes to the Agent class
                self.grid.place_agent(sugar, (x,y))
                self.schedule.add(sugar)
                # print(self.schedule.agents_by_type[Sugar][agent_id])
                agent_id+=1
                # print(sugar.unique_id, sugar.pos, sugar.max_sugar)
        
            max_spice = spice_distribution[x,y]
            if max_spice > 0:
                spice = Spice(agent_id, self, (x,y), max_spice)
                self.grid.place_agent(spice, (x,y))
                self.schedule.add(spice)
                # print(self.schedule.agents_by_type[Spice][agent_id])
                agent_id+=1
        
        for i in range(self.initial_population):
            # Get agent position
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            # See Growing Artificial Society p.108 for initialization
            # Give agents initial endowment
            sugar = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
            spice = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
            # Give agents initial metabolism
            metabolism_sugar = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
            metabolism_spice = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
            # Give agents vision
            vision = int(self.random.uniform(self.vision_min, self.vision_max+1))
            # Create Trader object
            trader = Trader(agent_id, 
                            self, 
                            (x,y), 
                            moore=False, 
                            sugar=sugar,
                            spice=spice,
                            metabolism_sugar=metabolism_sugar,
                            metabolism_spice=metabolism_spice,
                            vision=vision)
            # Place agent
            self.grid.place_agent(trader, (x,y))
            self.schedule.add(trader)
            agent_id +=1

    def step(self):
        '''
        Unique step function that does staged activation of sugar and spice and then randomly activates traders.
        '''

        for sugar in self.schedule._agents_by_type[Sugar]:
            sugar.step()
        for spice in self.schedule._agents_by_type[Spice]:
            spice.step()


        self.schedule.steps += 1 # Important for Data Collector to track the number of steps
        print(self.schedule.steps, self.schedule.time)
    
    def run_model(self, step_count=1000):

        for i in range(step_count):
            self.step()
            




In [126]:
model = SugarscapeG1mt()

model.step()


1 0
